## Custom Lookup Table Demo

This notebook demonstrates **custom lookup tables** in EZKL: using a user-defined piecewise-linear (PWL) table instead of the built-in Sigmoid lookup. This is useful when you need a custom approximation (e.g. different precision or range) for non-linear activations.

### How to produce the lookup table

The lookup table is a JSON file with three arrays:
- **breakpoints** (length n+1): strictly increasing x-values defining segment boundaries. Segments are `[breakpoints[i], breakpoints[i+1])`.
- **slopes** (length n): for segment i, the linear map is `y = slopes[i] * x + intercepts[i]`.
- **intercepts** (length n): intercept for each segment.

You can generate this file by:
1. Choosing breakpoints that cover your model's activation range (e.g. for sigmoid, a symmetric range like [-6, 6]). Breakpoints can be **uniform** (evenly spaced) or **non-uniform** (e.g. more segments where the function has high curvature—see `generate_pwl_json` with `spacing="curvature"`).
2. Computing the target function (e.g. sigmoid) at each breakpoint.
3. For each segment [a, b], set slope = (f(b)-f(a))/(b-a) and intercept = f(a) - slope*a.

The helper **`generate_pwl_json`** supports: (a) custom breakpoints (you pass a list); (b) uniform spacing; (c) curvature-based spacing (denser where the function bends more); (d) **quantile-based** (`spacing="quantile"`, pass `data=...`), which adapts segments to your input distribution (production-style). You can use many segments (e.g. 1024) for a high-resolution lookup table.

**Caveat:** The PWL must approximate the same function your model expects (e.g. sigmoid). Large deviation can cause soundness issues or proof failure; keep approximation error within your tolerance. **Input must be within the defined breakpoints**—the circuit will error if the lookup range extends outside your PWL range (see docs).

With a small number of segments (e.g. 4–5) over a range like `[-6, 6]`, a well-built PWL can achieve **max |approx − σ(x)| on the order of 1e-11** vs the true sigmoid, which is a strong usability and accuracy guarantee.

**Production tip:** If you set `lookup_range` explicitly (e.g. with a safety margin for runtime values), extend your PWL breakpoints to cover that range so the circuit's lookup range stays **within** the table (required by the input-range check). For example, build the PWL over `[x_min - margin, x_max + margin]` and set `lookup_range` to match.

In [ ]:
import json
import math
import os

def generate_pwl_json(f, range_lo=None, range_hi=None, num_segments=None, breakpoints=None, spacing="uniform", data=None, out_path=None):
    """
    Build a PWL lookup table for a scalar function f.

    Supports four ways to define segments:
    1) Custom breakpoints: pass breakpoints=[x0, x1, ..., xN] (must be strictly increasing).
    2) Uniform spacing: pass range_lo, range_hi, num_segments; breakpoints are evenly spaced.
    3) Curvature-based: pass range_lo, range_hi, num_segments and spacing="curvature";
       more segments where |f''| is large (e.g. near 0 for sigmoid).
    4) Quantile-based (data-driven): pass data=(array of x values), num_segments, and spacing="quantile";
       breakpoints at quantiles of data so segments adapt to input distribution (production-style).

    Returns the dict {breakpoints, slopes, intercepts}; if out_path is set, also writes JSON.
    Production: if you set lookup_range with a margin, extend PWL breakpoints to cover that range.
    """
    if breakpoints is not None:
        # User-provided breakpoints (must be strictly increasing)
        assert len(breakpoints) >= 2 and breakpoints == sorted(breakpoints), "breakpoints must be strictly increasing"
        bp = [float(x) for x in breakpoints]
    elif spacing == "quantile" and data is not None and num_segments is not None:
        # Data-driven: breakpoints at quantiles of data (like production pipelines)
        sorted_data = sorted(data)
        n_data = len(sorted_data)
        if n_data < 2:
            raise ValueError("quantile spacing requires at least 2 data points")
        indices = [int(i * (n_data - 1) / num_segments) for i in range(num_segments + 1)]
        indices[-1] = n_data - 1
        bp = [float(sorted_data[i]) for i in indices]
        bp = sorted(set(bp))
        if len(bp) < 2:
            bp = [float(min(data)), float(max(data))]
    elif spacing == "curvature" and num_segments is not None and range_lo is not None and range_hi is not None:
        # Place breakpoints by function curvature: more points where |f''| is large (e.g. near 0 for sigmoid)
        fine = 4 * max(num_segments, 256)
        xs = [range_lo + (range_hi - range_lo) * i / fine for i in range(fine + 1)]
        h = (range_hi - range_lo) / fine
        curv = [max(abs(f(xs[i] + h) - 2 * f(xs[i]) + f(xs[i] - h)) / (h * h) if h > 0 else 0, 1e-20) for i in range(1, fine)]
        cum = [0.0]
        for c in curv:
            cum.append(cum[-1] + c)
        total = cum[-1]
        bp = [float(range_lo)]
        for k in range(1, num_segments):
            target = k * total / num_segments
            for i in range(1, len(cum)):
                if cum[i] >= target:
                    if cum[i] > cum[i - 1] and i + 1 < len(xs):
                        t = (target - cum[i - 1]) / (cum[i] - cum[i - 1])
                        x = xs[i] + t * (xs[i + 1] - xs[i])
                    else:
                        x = xs[min(i + 1, len(xs) - 1)]
                    if x > bp[-1]:
                        bp.append(float(x))
                    break
        bp.append(float(range_hi))
        bp = sorted(set(bp))  # strictly increasing, dedupe
    else:
        # Uniform spacing
        assert range_lo is not None and range_hi is not None and num_segments is not None
        bp = [range_lo + (range_hi - range_lo) * i / num_segments for i in range(num_segments + 1)]

    n = len(bp) - 1
    slopes, intercepts = [], []
    for i in range(n):
        a, b = bp[i], bp[i + 1]
        fa, fb = f(a), f(b)
        slope = (fb - fa) / (b - a) if b != a else 0.0
        intercept = fa - slope * a
        slopes.append(slope)
        intercepts.append(intercept)
    pwl = {"breakpoints": bp, "slopes": slopes, "intercepts": intercepts}
    if out_path:
        with open(out_path, 'w') as fp:
            json.dump(pwl, fp, indent=2)
    return pwl

# Example 1: sigmoid PWL on [-6, 6] with 4 uniform segments (fast demo, same as pwl_sigmoid_example.json)
sigmoid = lambda x: 1.0 / (1.0 + math.exp(-x))
pwl_path = 'pwl_sigmoid.json'
generate_pwl_json(sigmoid, range_lo=-6.0, range_hi=6.0, num_segments=4, out_path=pwl_path)
print('Saved PWL table to', pwl_path)

# Example 2 (optional): 1024 segments with curvature-based breakpoints (denser near 0).
# generate_pwl_json(sigmoid, range_lo=-6.0, range_hi=6.0, num_segments=1024, spacing="curvature", out_path="pwl_sigmoid_1024.json")

# Example 3 (optional): quantile-based from data (production-style). Uncomment and set your data.
# sample_data = [0.1, -0.5, 0.3, ...]  # e.g. from calibration or input_data
# generate_pwl_json(sigmoid, num_segments=64, spacing="quantile", data=sample_data, out_path="pwl_sigmoid_quantile.json")

In [ ]:
try:
    import google.colab
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ezkl"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "onnx"])
except Exception:
    pass

from torch import nn
import ezkl
import os
import json
import torch

class SigmoidModel(nn.Module):
    def __init__(self):
        super(SigmoidModel, self).__init__()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return self.sigmoid(x)

circuit = SigmoidModel()

In [ ]:
model_path = os.path.join('network.onnx')
compiled_model_path = os.path.join('network.compiled')
pk_path = os.path.join('test.pk')
vk_path = os.path.join('test.vk')
settings_path = os.path.join('settings.json')
witness_path = os.path.join('witness.json')
data_path = os.path.join('input.json')
pwl_path = os.path.abspath('pwl_sigmoid.json')

In [ ]:
shape = [3]
x = 0.5 * (torch.rand(1, *shape) - 0.5)
circuit.eval()
torch.onnx.export(
    circuit, x, model_path,
    export_params=True, opset_version=10, do_constant_folding=True,
    input_names=['input'], output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)
data = dict(input_data=[x.detach().numpy().reshape(-1).tolist()])
json.dump(data, open(data_path, 'w'))

In [ ]:
py_run_args = ezkl.PyRunArgs()
py_run_args.input_visibility = "public"
py_run_args.output_visibility = "public"
py_run_args.param_visibility = "fixed"
py_run_args.custom_lookup_path = pwl_path

res = ezkl.gen_settings(model_path, settings_path, py_run_args=py_run_args)
assert res is True

In [ ]:
cal_path = os.path.join('calibration.json')
cal_data = dict(input_data=[torch.rand(20, *shape).detach().numpy().reshape(-1).tolist()])
json.dump(cal_data, open(cal_path, 'w'))
ezkl.calibrate_settings(cal_path, model_path, settings_path, 'resources')

In [ ]:
res = ezkl.compile_circuit(model_path, compiled_model_path, settings_path)
assert res is True

In [ ]:
res = ezkl.get_srs(settings_path)

In [ ]:
res = ezkl.gen_witness(data_path, compiled_model_path, witness_path)
assert os.path.isfile(witness_path)

In [ ]:
res = ezkl.setup(compiled_model_path, vk_path, pk_path)
assert res is True
assert os.path.isfile(vk_path)
assert os.path.isfile(pk_path)

In [ ]:
proof_path = os.path.join('test.pf')
res = ezkl.prove(witness_path, compiled_model_path, pk_path, proof_path)
assert os.path.isfile(proof_path)

In [ ]:
res = ezkl.verify(proof_path, settings_path, vk_path)
assert res is True
print('Verified.')